In [ ]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict

In [2]:
load_dotenv()

model = ChatGroq(model="llama-3.1-8b-instant")


In [3]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str


In [4]:
def calculate_sr(state: BatsmanState):
    sr = (state["runs"] / state["balls"]) * 100

    return {"sr": sr}


def calculate_bpb(state: BatsmanState):
    bpb = state["balls"] / (state["fours"] + state["sixes"])

    return {"bpb": bpb}


def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = (
                               ((state["fours"] * 4) + (state["sixes"] * 6)) / state["runs"]
                       ) * 100

    return {"boundary_percent": boundary_percent}


def summary(state: BatsmanState):
    summary = f"""
Strike Rate - {state["sr"]} \n
Balls per boundary - {state["bpb"]} \n
Boundary percent - {state["boundary_percent"]}
"""

    return {"summary": summary}

In [5]:
graph = StateGraph(BatsmanState)

graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percent", calculate_boundary_percent)
graph.add_node("summary", summary)

# edges

graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percent")

graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_boundary_percent", "summary")

graph.add_edge("summary", END)

workflow = graph.compile()


In [6]:
intial_state = {"runs": 100, "balls": 50, "fours": 6, "sixes": 4}

workflow.invoke(intial_state)


{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundary_percent': 48.0,
 'summary': '\nStrike Rate - 200.0 \n\nBalls per boundary - 5.0 \n\nBoundary percent - 48.0\n'}

In [7]:
workflow.get_graph().draw_mermaid_png()

b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x02O\x00\x00\x01M\x08\x02\x00\x00\x00\x96\x85\x1c\xaf\x00\x00\x10\x00IDATx\x9c\xec\xdd\x07`\x13\xd5\x1f\x07\xf0wY\xdd\x0b:(\xa5\xb4\xec){/A(\xc8\x06E@\x86\n(\xa0"SA@\x04QA\x11D\x01\x17"*"\x82\n\x02\xf2W\x10\x11\x10\x19\nj\xd9\x08m\xd9\x94\xd5\xbd3\xff\xbf\xe4\xe8\x91\xa6I\xda\xa6M{\x97|?\x7f\xfe\xf1r3\xbd{\xf7~\xf7\xde\xbb{\xa70\x18\x0c\x0c\x00\x00\xc0\xa5)\x18\x00\x00\x80\xabC\xb4\x03\x00\x00\xd7\x87h\x07\x00\x00\xae\x0f\xd1\x0e\x00\x00\\\x1f\xa2\x1d\x00\x00\xb8>D;\x00\x00p}\x88v\x00\xd2\x96rG}\xf2\xf7\xd4\xa4\x9b\xea\xdcl\xbdA\xcb4Z\xbd0\x89c\x8c\x7f\xc0H.\xe7t:\xe3\xa0\x8ccz\xd3(\xb9\x8c\xe9L3r\x1c\xe3\x9fB\x12&)\x14\x9cV{\xff\xc1$~<\xc7q\xc2\xd3J\xc2\x9c\x05\xb6a\xdc\x8aL\xa7\xd3s2\xce\xa07\x98\x8d6.K_\xf5f\x0f;\xa9T\x9cL\xce<}\xe4\xa1\xd5=Zv\x0fPy\xaa\x18\x80\x93qx\xde\x0e@\x8an_\xcf\xfae\xdd\xed\xb4;::\x83\xe5J\xe6\xe5+W(e2\x8e\xd3\xaa\x8dg\xb4\x81Q\xbc\xa1\xff\x99\xfeK\xf1I\xc1\xe9\xf9\x00&30\xbdq\x0c\xa7`\x14\x1a\x8ds\x9a\xe62\xca\x